In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import matplotlib as mpl

mpl.rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False

# 자연어 변환 함수
def shift_to_korean(t):
    if t == 1:
        return "내일"
    elif t == 2:
        return "모레"
    else:
        return f"{t}일 후"

# 1. 데이터 불러오기
df = pd.read_csv("v3_news_2020_2025_06_combined.csv")
df['adjusted_date'] = pd.to_datetime(df['adjusted_date'], errors='coerce')
df = df.sort_values(by=['industry', 'adjusted_date'])

# 2. 파생 피처 생성
df['industry_avg_close'] = df.groupby(['industry', 'adjusted_date'])['close'].transform('mean')
df['industry_avg_foreign_vol'] = df.groupby(['industry', 'adjusted_date'])['foreign_vol'].transform('mean')
df['industry_avg_volume'] = df.groupby(['industry', 'adjusted_date'])['volume'].transform('mean')
df['score_change'] = df.groupby('industry')['normalized_score'].diff()
df['score_ma3'] = df.groupby('industry')['normalized_score'].rolling(3).mean().reset_index(0, drop=True)
df['score_ma5'] = df.groupby('industry')['normalized_score'].rolling(5).mean().reset_index(0, drop=True)
df['volatility'] = df.groupby('industry')['industry_avg_close'].rolling(3).std().reset_index(0, drop=True)

# 3. manual_weights
manual_weights = {
    '항공운송': {'normalized_score': 0.7, 'score_ma3': 0.1, 'volatility': 0.1, 'score_change': 0.1},
    '전자부품': {'normalized_score': 0.4, 'industry_avg_foreign_vol': 0.3, 'industry_avg_volume': 0.3},
    '바이오': {'score_ma5': 0.5, 'normalized_score': 0.3, 'volatility': 0.1, 'score_change': 0.1},
    '에너지': {'score_ma3': 0.5, 'normalized_score': 0.3, 'volatility': 0.2},
    '조선업': {'score_change': 0.4, 'normalized_score': 0.4, 'score_ma3': 0.2},
    '금속제조업': {'score_ma3': 0.3, 'score_change': 0.3, 'volatility': 0.2, 'normalized_score': 0.2},
    '반도체제조업': {'foreign_ratio': 0.3, 'score_change': 0.2, 'volatility': 0.2, 'score_ma3': 0.3},
    '자동차': {'normalized_score': 0.2, 'score_change': 0.6, 'score_ma3': 0.2},
    '정보통신업': {'normalized_score': 0.1, 'score_ma5': 0.8, 'score_change': 0.1},
    '건설업': {'normalized_score': 0.1, 'volatility': 0.1, 'score_ma5': 0.8},
    '석유정제': {'normalized_score': 0.2, 'score_ma5': 0.8},
    '금융업': {'normalized_score': 0.5, 'volume': 0.2, 'institution_vol': 0.15, 'foreign_vol': 0.15},
    '보험업': {'normalized_score': 0.3, 'volume': 0.25, 'institution_vol': 0.2, 'foreign_ratio': 0.15, 'foreign_vol': 0.15},
    '식료품': {'normalized_score': 0.2, 'volume': 0.2, 'institution_vol': 0.1, 'foreign_vol': 0.1, 'foreign_ratio': 0.1, 'volatility': 0.1, 'score_change': 0.1, 'score_ma3': 0.1},
    '방송업': {'normalized_score': 0.7, 'score_change': 0.3}
}

# 4. 산업군별 모델 선택
def get_model(industry, y_train=None):
    if industry in ['자동차', '정보통신업', '건설업', '석유정제', '금융업']:
        pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
        return CatBoostClassifier(random_seed=42, verbose=0, class_weights=[1, pos_weight])
    else:
        return LGBMClassifier(random_state=42, is_unbalance=True)

# 5. 시차 설정
def get_t_shift(industry):
    return {'전자부품': 3, '정보통신업': 3}.get(industry, 1)

# 6. 예측 결과 저장
results = []

for industry, weights in manual_weights.items():
    features = list(weights.keys())
    sub = df[df['industry'] == industry].copy()

    sub = sub.dropna(subset=features)
    if sub.empty:
        continue

    t_shift = get_t_shift(industry)
    sub['future_close'] = sub['industry_avg_close'].shift(-t_shift)
    sub['log_return'] = np.log(sub['future_close'] / sub['industry_avg_close'])
    sub['label'] = (sub['log_return'] > 0.01).astype(int)

    train = sub[sub['adjusted_date'] < '2025-06-01']
    test = sub[sub['adjusted_date'] == '2025-06-01']

    if train.empty or test.empty:
        continue

    X_train = train[features]
    y_train = train['label']
    X_test = test[features]

    model = get_model(industry, y_train)
    model.fit(X_train, y_train)

    probas = model.predict_proba(X_test)[:, 1]
    avg_proba = np.mean(probas)

    results.append({
        'industry': industry,
        'predicted_prob': avg_proba * 100
    })

# 7. optimal_buy_data 정의
optimal_buy_data = {
    "industry": [
        "건설업", "금속제조업", "금융업", "바이오", "반도체제조업", "방송업", "보험업",
        "석유정제", "식료품", "에너지", "자동차", "전자부품", "정보통신업", "조선업", "항공운송"
    ],
    "T_buy": [
        0.1, 0.7, 0.4, 0.5, 0.2, 0.1, 0.2, 0.4, 0.1, 0.4, 0.6, 0.2, 0.1, 0.2, 0.4
    ]
}
optimal_df = pd.DataFrame(optimal_buy_data)
pred_df = pd.DataFrame(results)

# 8. 병합 및 시각화용 컬럼 생성
merged = pd.merge(optimal_df, pred_df, on='industry', how='left')
merged['predicted_prob'] = merged['predicted_prob'].fillna(0)

# 9. 산업군별 개별 표 생성 (t_shift 반영)
for _, row in merged.iterrows():
    industry = row['industry']
    pred_prob = row['predicted_prob']
    T_buy = row['T_buy']

    # 산업군별 t_shift
    t_shift = get_t_shift(industry)
    update_date = pd.to_datetime("2025-06-01")
    target_date = update_date + pd.Timedelta(days=t_shift)

    shift_text = shift_to_korean(t_shift)
    예측대상일 = f"{shift_text}({target_date.strftime('%m.%d')})"

    상승확률 = f"{pred_prob:.1f}%" if pred_prob > 0 else '–'
    매수기준 = f"{T_buy * 100:.1f}%"
    충족여부 = '충족' if pred_prob > (T_buy * 100) else '미충족'

    table_data = [[industry,
                   update_date.strftime("%Y-%m-%d"),
                   예측대상일,
                   상승확률,
                   매수기준,
                   충족여부]]

    col_labels = ["산업군", "업데이트 날짜", "예측 대상일", f"{shift_text} 상승 확률", "매수 추천 기준", "기준 충족 여부"]

    fig, ax = plt.subplots(figsize=(8, 2.5))
    ax.axis('off')

    tbl = ax.table(cellText=table_data,
                   colLabels=col_labels,
                   colColours=['#e8f0fe'] * len(col_labels),
                   cellLoc='center',
                   loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(11)
    tbl.scale(1.2, 1.5)

    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_text_props(weight='bold', color='black')
        else:
            cell.set_facecolor('white')
        cell.set_linewidth(1)

    plt.tight_layout()
    # 저장하려면 이 주석 해제
    # plt.savefig(f"{industry}_{update_date}_예측결과표_.png", dpi=300)
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import matplotlib as mpl

mpl.rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False

# 자연어 변환 함수
def shift_to_korean(t):
    if t == 1:
        return "내일"
    elif t == 2:
        return "모레"
    else:
        return f"{t}일 후"

#  데이터 입력
train_df = pd.read_csv("train파일.csv") #기존
test_df = pd.read_csv("test파일.csv") #실시간

train_df['adjusted_date'] = pd.to_datetime(train_df['adjusted_date'], errors='coerce')
test_df['adjusted_date'] = pd.to_datetime(test_df['adjusted_date'], errors='coerce')

train_df = train_df.sort_values(by=['industry', 'adjusted_date'])
test_df = test_df.sort_values(by=['industry', 'adjusted_date'])

# 파생 피처 생성
for df in [train_df, test_df]:
    df['industry_avg_close'] = df.groupby(['industry', 'adjusted_date'])['close'].transform('mean')
    df['industry_avg_foreign_vol'] = df.groupby(['industry', 'adjusted_date'])['foreign_vol'].transform('mean')
    df['industry_avg_volume'] = df.groupby(['industry', 'adjusted_date'])['volume'].transform('mean')
    df['score_change'] = df.groupby('industry')['normalized_score'].diff()
    df['score_ma3'] = df.groupby('industry')['normalized_score'].rolling(3).mean().reset_index(0, drop=True)
    df['score_ma5'] = df.groupby('industry')['normalized_score'].rolling(5).mean().reset_index(0, drop=True)
    df['volatility'] = df.groupby('industry')['industry_avg_close'].rolling(3).std().reset_index(0, drop=True)

# manual_weights
manual_weights = {
    '항공운송': {'normalized_score': 0.7, 'score_ma3': 0.1, 'volatility': 0.1, 'score_change': 0.1},
    '전자부품': {'normalized_score': 0.4, 'industry_avg_foreign_vol': 0.3, 'industry_avg_volume': 0.3},
    '바이오': {'score_ma5': 0.5, 'normalized_score': 0.3, 'volatility': 0.1, 'score_change': 0.1},
    '에너지': {'score_ma3': 0.5, 'normalized_score': 0.3, 'volatility': 0.2},
    '조선업': {'score_change': 0.4, 'normalized_score': 0.4, 'score_ma3': 0.2},
    '금속제조업': {'score_ma3': 0.3, 'score_change': 0.3, 'volatility': 0.2, 'normalized_score': 0.2},
    '반도체제조업': {'foreign_ratio': 0.3, 'score_change': 0.2, 'volatility': 0.2, 'score_ma3': 0.3},
    '자동차': {'normalized_score': 0.2, 'score_change': 0.6, 'score_ma3': 0.2},
    '정보통신업': {'normalized_score': 0.1, 'score_ma5': 0.8, 'score_change': 0.1},
    '건설업': {'normalized_score': 0.1, 'volatility': 0.1, 'score_ma5': 0.8},
    '석유정제': {'normalized_score': 0.2, 'score_ma5': 0.8},
    '금융업': {'normalized_score': 0.5, 'volume': 0.2, 'institution_vol': 0.15, 'foreign_vol': 0.15},
    '보험업': {'normalized_score': 0.3, 'volume': 0.25, 'institution_vol': 0.2, 'foreign_ratio': 0.15, 'foreign_vol': 0.15},
    '식료품': {'normalized_score': 0.2, 'volume': 0.2, 'institution_vol': 0.1, 'foreign_vol': 0.1, 'foreign_ratio': 0.1, 'volatility': 0.1, 'score_change': 0.1, 'score_ma3': 0.1},
    '방송업': {'normalized_score': 0.7, 'score_change': 0.3}
}

# 산업군별 모델 선택
def get_model(industry, y_train=None):
    if industry in ['자동차', '정보통신업', '건설업', '석유정제', '금융업']:
        pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
        return CatBoostClassifier(random_seed=42, verbose=0, class_weights=[1, pos_weight])
    else:
        return LGBMClassifier(random_state=42, is_unbalance=True)

# 시차 설정
def get_t_shift(industry):
    return {'전자부품': 3, '정보통신업': 3}.get(industry, 1)

# 예측 결과 저장
results = []

for industry, weights in manual_weights.items():
    features = list(weights.keys())

    train = train_df[train_df['industry'] == industry].copy().dropna(subset=features)
    test = test_df[test_df['industry'] == industry].copy().dropna(subset=features)

    if train.empty or test.empty:
        continue

    t_shift = get_t_shift(industry)
    train['future_close'] = train['industry_avg_close'].shift(-t_shift)
    train['log_return'] = np.log(train['future_close'] / train['industry_avg_close'])
    train['label'] = (train['log_return'] > 0.01).astype(int)

    X_train = train[features]
    y_train = train['label']
    X_test = test[features]

    model = get_model(industry, y_train)
    model.fit(X_train, y_train)

    probas = model.predict_proba(X_test)[:, 1]
    avg_proba = np.mean(probas)

    results.append({
        'industry': industry,
        'predicted_prob': avg_proba * 100
    })

# optimal_buy_data 정의
optimal_buy_data = {
    "industry": [
        "건설업", "금속제조업", "금융업", "바이오", "반도체제조업", "방송업", "보험업",
        "석유정제", "식료품", "에너지", "자동차", "전자부품", "정보통신업", "조선업", "항공운송"
    ],
    "T_buy": [
        0.1, 0.7, 0.4, 0.5, 0.2, 0.1, 0.2, 0.4, 0.1, 0.4, 0.6, 0.2, 0.1, 0.2, 0.4
    ]
}
optimal_df = pd.DataFrame(optimal_buy_data)
pred_df = pd.DataFrame(results)

# 병합 및 시각화용 컬럼 생성
merged = pd.merge(optimal_df, pred_df, on='industry', how='left')
merged['predicted_prob'] = merged['predicted_prob'].fillna(0)

# 산업군별 표 시각화
for _, row in merged.iterrows():
    industry = row['industry']
    pred_prob = row['predicted_prob']
    T_buy = row['T_buy']

    t_shift = get_t_shift(industry)
    update_date = pd.to_datetime(test_df['adjusted_date'].max())
    target_date = update_date + pd.Timedelta(days=t_shift)

    shift_text = shift_to_korean(t_shift)
    예측대상일 = f"{shift_text}({target_date.strftime('%m.%d')})"

    상승확률 = f"{pred_prob:.1f}%" if pred_prob > 0 else '–'
    매수기준 = f"{T_buy * 100:.1f}%"
    충족여부 = '충족' if pred_prob > (T_buy * 100) else '미충족'

    table_data = [[industry,
                   update_date.strftime("%Y-%m-%d"),
                   예측대상일,
                   상승확률,
                   매수기준,
                   충족여부]]

    col_labels = ["산업군", "업데이트 날짜", "예측 대상일", f"{shift_text} 상승 확률", "매수 추천 기준", "기준 충족 여부"]

    fig, ax = plt.subplots(figsize=(8, 2.5))
    ax.axis('off')

    tbl = ax.table(cellText=table_data,
                   colLabels=col_labels,
                   colColours=['#e8f0fe'] * len(col_labels),
                   cellLoc='center',
                   loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(11)
    tbl.scale(1.2, 1.5)

    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_text_props(weight='bold', color='black')
        else:
            cell.set_facecolor('white')
        cell.set_linewidth(1)

    plt.tight_layout()
    # 저장하려면 이 부분 주석 해제
    # plt.savefig(f"{industry}_{update_date}_예측결과표.png", dpi=300)
    plt.show()
